In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

In [ ]:
df=pd.read_parquet(r"D:\AI\Real projects\Academic_Advisor\data\features\merged_add_acd_crg.parquet")

In [ ]:
df.head()

In [ ]:
df[df['student_id']==10428.111]

In [ ]:
df.columns

In [ ]:
# df_dropped=df.drop(columns=['student_course_id', 'student_id', 'course_id','degree_id',
#        'faculty_id','student_status_id','crg_degree_id',])

In [ ]:
diagnostic_cols = [
    "crg_degree_id", "acd_degree_ids", "acd_degree_count",
    "issue_type", "resolved_acd_degree_id", "recommended_action",
    "student_degree_id_original", "degree_id_for_acd_merge",
    "degree_id_corrected_for_acd", "acd_resolution_rule",
    "acd_degree_id", "degree_course_id",
    "acd_course_name_sl", "acd_degree_name_sl",  # duplicates of course_name_sl, degree_name_sl
]
df1 = df.drop(columns=[c for c in diagnostic_cols if c in df])
# 55 cols → ~41 cols

In [ ]:
df1.columns

In [ ]:
df1 = df1.dropna(subset=['start_agpa_points'])

In [ ]:
df1.info()

In [ ]:
FEATURE_COLS_BASELINE = [
    "prev_gpa_points",
    "prev_gpa_percent",
    "start_agpa_points",
    "start_agpa_percent",
    "registered_academic_semesters",
    "inactive_semester_gap",
    "has_inactive_semester_gap",
    "start_level_id",
    "degree_id",
    "course_id",
    "semester_num",
    "requirement_type_id_model",
    "course_credits_model",
    "degree_requirement_credits_count_model",
    "has_degree_course_info",
    "has_trusted_acd_features",
    "acd_match_type",
    "attempt_number",
    "in_credits",
    "in_gpa",
    "in_agpa",
]

In [ ]:
df1.head()

In [ ]:
df1[(df1['student_id'].eq('10000.111')&df1['part_id'].eq(20151))]

In [ ]:
# drop has_add_snapshot start_level_id study mode acd_course_credits acd_match_type student_course_id student_id faculty_id grade_id

In [ ]:
df_feature=df1.drop(columns=['has_add_snapshot','start_level_id','study_mode','acd_course_credits','acd_match_type'])
# ','student_course_id','student_id','faculty_id','grade_id','student_status_id'

In [ ]:
df1['has_add_snapshot'].value_counts()

In [ ]:
df1['start_level_id'].value_counts()

In [ ]:
df1['has_degree_course_info'].value_counts()

In [ ]:
import numpy as np

In [ ]:
df1['in_credits'].value_counts()

In [ ]:
df1['in_gpa'].value_counts()

In [ ]:
df1.head()

In [ ]:
df_feature.head()

In [ ]:
#register_status  in_credits	in_gpa	in_agpa	course_name_sl	degree_name_sl	 finish_status  attempt_count requirement_type_sl has_degree_course_info finish_part_id course_outcome_status

In [ ]:
#points  finish_status finish_status_add_snapshot finish_part_id

In [ ]:
df_feature2=df_feature.drop(columns=['register_status','in_credits','in_gpa','in_agpa','course_name_sl','degree_name_sl','finish_status','attempt_count','requirement_type_sl','has_degree_course_info','finish_part_id','finish_status_add_snapshot','points','course_outcome_status'])

In [ ]:
df_feature.info()

In [ ]:
df_feature['register_status'].value_counts()

In [ ]:
df_feature2.head()

In [ ]:
##prev_gpa_percent start_agpa_percent total_semesters

In [ ]:
df_feature2 = df_feature2.drop(columns=['prev_gpa_percent', 'start_agpa_percent', 'total_semesters'])

In [ ]:
df_feature2.head()

In [ ]:
df_feature2.describe()

In [ ]:
df_feature2[df_feature2['total_fail_credits'].eq(713.5)]

In [ ]:
df_feature2.info()

In [ ]:
df_feature2.isna().sum()

In [ ]:
df_feature2[df_feature2['course_credits'].eq(24)]

In [ ]:
# degree_id course_id ## combine

In [ ]:
# part_id # split start_part_id

In [ ]:
# requirement_type_id	degree_requirement_credits_count?

In [ ]:
## ordinry incoding 

In [ ]:
# final mark drop 

In [ ]:
a=df_feature2['semester_reg_credits'].value_counts()

In [ ]:
a[1].max()

In [ ]:
df_feature2[df_feature2['semester_reg_credits'].eq(72.5)]

In [ ]:
df_feature2.describe()

In [ ]:
df_feature2.info()

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

# # تفعيل الطابع البصري الأنيق للمخططات
# sns.set_theme(style="whitegrid")

# # قائمة الأعمدة المراد فحص قيمها الشاذة وتوزيعها
# columns_to_analyze = ['course_credits', 'semester_reg_credits','attempt_number', 'total_fail_credits','total_pass_credits','start_total_in_credits','semester_reg_courses','reg_total_semesters']

# for col in columns_to_analyze:
#     # 1. حساب الأرقام الدقيقة للقيم الشاذة باستخدام IQR
#     q1 = df_feature2[col].quantile(0.25)
#     q3 = df_feature2[col].quantile(0.75)
#     iqr = q3 - q1
#     lower_bound = q1 - 1.5 * iqr
#     upper_bound = q3 + 1.5 * iqr
    
#     # تحديد الحسابات الشاذة
#     outliers = df_feature2[(df_feature2[col] < lower_bound) | (df_feature2[col] > upper_bound)]
#     num_outliers = len(outliers)
#     pct_outliers = (num_outliers / len(df_feature2)) * 100
    
#     # 2. إنشاء الشكل ذو اللوحتين (Two Plots)
#     fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [2, 1]})
#     fig.suptitle(f"Analysis for Column: '{col}'", fontsize=16, fontweight='bold', y=1.02)
    
#     # اللوحة الأولى (يسار): مخطط التوزيع ورؤية التكتلات (KDE + Histogram)
#     sns.histplot(data=df_feature2, x=col, kde=True, ax=axes[0], color='darkblue', bins=50)
#     axes[0].set_title("Data Distribution & Grouping Clusters", fontsize=12)
#     axes[0].set_xlabel(col)
#     axes[0].set_ylabel("Count")
    
#     # اللوحة الثانية (يمين): مخطط الصندوق لإظهار النقاط الشاذة بوضوح (Boxplot)
#     sns.boxplot(data=df_feature2, y=col, ax=axes[1], color='darkred', flierprops={"marker": "x", "markerfacecolor": "red"})
#     axes[1].set_title("Boxplot (Outliers Visualized as 'x')", fontsize=12)
#     axes[1].set_ylabel(col)
    
#     # إضافة نص توضيحي على الرسم يبين عدد القيم الشاذة وحدود المنطق
#     info_text = (
#         f"Total Rows: {len(df_feature2):,}\n"
#         f"Lower Bound: {lower_bound:.2f}\n"
#         f"Upper Bound: {upper_bound:.2f}\n"
#         f"Outliers Count: {num_outliers:,}\n"
#         f"Outliers Pct: {pct_outliers:.2f}%"
#     )
#     axes[1].text(1.05, 0.5, info_text, transform=axes[1].transAxes, fontsize=11,
#                  bbox=dict(boxstyle="round,pad=0.5", facecolor="wheat", alpha=0.5))
    
#     plt.tight_layout()
#     plt.show()
    
#     # طباعة التقرير في واجهة الكود (Console)
#     print(f"=== Report for {col} ===")
#     print(f"Number of Outliers: {num_outliers:,} out of {len(df_feature2):,} ({pct_outliers:.2f}%)")
#     print(f"Healthy Range: [{lower_bound} to {upper_bound}]\n" + "="*40 + "\n")

In [ ]:
df_feature2.corr(numeric_only=True)

In [ ]:
df_feature2['prev_gpa_points'].value_counts(dropna=False)

In [ ]:
d=pd.read_parquet(r'D:\AI\Real projects\Academic_Advisor\data\raw\v_add_student_degree_status.parquet')

In [ ]:
d[d['student_id'].eq(130.111)].head(10)

In [ ]:
d[d['start_agpa_points'].eq(0)]

In [ ]:
a=d[(d['part_id'].eq(20164)) & (d['semester_in_credits'] > 24)]

In [ ]:
len(a)

In [ ]:
a.head()

In [ ]:
mask = d["semester_reg_credits"].gt(25)

print("Rows with semester_reg_credits > 24:", mask.sum())
print("Unique students:", d.loc[mask, "student_id"].nunique())

In [ ]:
d.loc[mask, ["student_id", "part_id", "degree_id", "semester_reg_credits"]].drop_duplicates()

In [ ]:
d1=pd.read_parquet(r'D:\AI\Real projects\Academic_Advisor\data\raw\v_crg_student_course_raw.parquet')

In [ ]:
d1[(d1['student_id'].eq(840.111))&(d1['part_id'].eq(20112.0))]

In [ ]:
df_feature2.to_parquet(r'D:\AI\Real projects\Academic_Advisor\data\audit\df_crg_add_acd.parquet')

## Pandas copy/view audit — select.ipynb

Two patterns fixed in this notebook:

| Cell | Before | After | Reason |
|---|---|---|---|
| `df1.dropna(...)` | `df1.dropna(subset=[...], inplace=True)` | `df1 = df1.dropna(subset=[...])` | Avoid inplace on df from `.drop()` — use assignment form |
| `df_feature2.drop(...)` | `df_feature2.drop(columns=[...], inplace=True)` | `df_feature2 = df_feature2.drop(columns=[...])` | Same — avoid inplace, use assignment form |

All other DataFrames in this notebook (`df1`, `df_feature`, `df_feature2`) are results of `.drop()` or `.read_parquet()` and are already independent copies. No `.copy()` additions were needed elsewhere.